In [ ]:
# En este código se descargan los .xml de data/bronze/boe_docs_xml

In [ ]:
import requests
import pandas as pd
from datetime import datetime, date, timezone, timedelta
import json
import time
import random
from typing import Any
import json
from pathlib import Path
import re

from renewables_permitting.utils import as_list, save_parquet, validate_required_columns

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

BOE_DOCS_XML_DIR = BRONZE_DIR / "boe_docs_xml"

In [ ]:
REQUIRED_XML_DOWNLOAD_COLS = {"identificador", "doc_file_stem", "url_xml"}

# Funciones

In [ ]:
def build_xml_path(output_dir: Path, doc_file_stem: str) -> Path:
    return output_dir / f"{doc_file_stem}.xml"

In [ ]:
def download_single_xml(
    identificador: str,
    doc_file_stem: str,
    url_xml: str,
    output_dir: Path,
    timeout: int = 30,
) -> dict:
    xml_path = build_xml_path(output_dir, doc_file_stem)

    record = {
        "identificador": identificador,
        "doc_file_stem": doc_file_stem,
        "url_xml": url_xml,
        "xml_path": str(xml_path),
        "download_status": None,
        "http_status_code": None,
        "downloaded_at": datetime.now(timezone.utc).isoformat(),
        "error_message": None,
        "file_size_bytes": None,
    }

    if pd.isna(url_xml) or not str(url_xml).strip():
        record["download_status"] = "missing_url"
        record["error_message"] = "missing url_xml"
        return record

    if xml_path.exists():
        record["download_status"] = "already_exists"
        record["file_size_bytes"] = xml_path.stat().st_size
        return record

    try:
        response = requests.get(url_xml, timeout=timeout)
        record["http_status_code"] = response.status_code

        if response.status_code != 200:
            record["download_status"] = "http_error"
            record["error_message"] = f"HTTP {response.status_code}"
            return record

        xml_path.write_bytes(response.content)

        record["download_status"] = "downloaded"
        record["file_size_bytes"] = xml_path.stat().st_size
        return record

    except requests.RequestException as exc:
        record["download_status"] = "request_error"
        record["error_message"] = str(exc)
        return record

In [ ]:
def download_boe_xml_documents(
    candidates_path: Path,
    output_dir: Path,
    log_path: Path | None = None,
    limit: int | None = None,
    timeout: int = 30,
) -> pd.DataFrame:
    output_dir.mkdir(parents=True, exist_ok=True)

    if log_path is None:
        log_path = output_dir / "download_log.parquet"

    boe_candidates = pd.read_parquet(candidates_path)

    if limit is not None:
        boe_candidates = boe_candidates.head(limit).copy()

    validate_required_columns(
        boe_candidates,
        REQUIRED_XML_DOWNLOAD_COLS,
    )

    download_records = []

    for row in boe_candidates.itertuples(index=False):
        record = download_single_xml(
            identificador=row.identificador,
            doc_file_stem=row.doc_file_stem,
            url_xml=row.url_xml,
            output_dir=output_dir,
            timeout=timeout,
        )

        download_records.append(record)

    new_log = pd.DataFrame(download_records)

    if log_path.exists():
        old_log = pd.read_parquet(log_path)
        full_log = pd.concat([old_log, new_log], ignore_index=True)
    else:
        full_log = new_log

    full_log.to_parquet(log_path, index=False)

    return new_log

# Prueba

In [ ]:
download_log = download_boe_xml_documents(
    candidates_path=SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet",
    output_dir=BOE_DOCS_XML_DIR
    #, limit=10,
)

In [ ]:
df = pd.read_parquet(BOE_DOCS_XML_DIR / "download_log.parquet")
df

In [ ]:
# TODO: Ahora mismo se ejecutan varias veces los archivos y en download_status aparece already_exists. 
# Los archivos descargados correctamente no deberían ser reprocesados.

df.loc[df["identificador"] == "BOE-B-2026-19619"]